# ISIC 2019 setup: tables and folders

This notebook creates the ISIC 2019 dataset-specific Bronze tables, the shared Silver tables, and (later) Gold tables, plus their storage folders.

Run this after `notebooks/00_setup_storage.ipynb` and before `10_bronze_ingest.ipynb`. It also prints the exact landing paths where the source archives must exist.


In [ ]:
%run ../_setup_env


In [ ]:
from data_platform.labels import SILVER_LABEL_COLUMN_DDL
from data_platform.layout import build_layout, load_dataset_config, load_yaml_config, resolve_archive_paths

DATASET_CONFIG = load_dataset_config(CONFIG_ROOT / "datasets" / "isic_2019.yaml")
STORAGE_CONFIG = load_yaml_config(CONFIG_ROOT / "storage.yaml")
DATASET_CONFIG["landing_root"] = STORAGE_CONFIG["landing_root"]
DATASET_CONFIG["storage_root"] = STORAGE_CONFIG["storage_root"]
layout = build_layout(DATASET_CONFIG)
landing_paths = layout["landing_paths"]
bronze_paths = layout["bronze_paths"]
bronze_tables = layout["bronze_tables"]
silver_tables = layout["silver_tables"]

ARCHIVES = resolve_archive_paths(DATASET_CONFIG["archives"], landing_paths, bronze_paths)

print(layout)
for archive in ARCHIVES:
    print(f"{archive['source_split']} archive upload path: {archive['archive_dbfs_path']}")
    print(f"{archive['source_split']} Bronze image rows table: {bronze_tables['image_blobs']}")
    print(f"{archive['source_split']} Bronze metadata path: {archive['metadata_target_path']}")


## Create Bronze Objects

Create the ISIC 2019 Bronze source metadata table, image blob table, and dataset-specific landing and metadata folders.


In [ ]:
spark.sql(
    """
    CREATE TABLE IF NOT EXISTS bronze.isic_2019_image_blobs (
      image_id STRING,
      source_split STRING,
      archive_member_path STRING,
      source_archive_uri STRING,
      image_bytes BINARY,
      byte_length BIGINT,
      source_checksum STRING
    )
    USING DELTA
    """
)
spark.sql(
    """
    CREATE TABLE IF NOT EXISTS bronze.isic_2019_source_metadata (
      image_id STRING,
      source_split STRING,
      source_uri STRING,
      attribution STRING,
      copyright_license STRING,
      age_approx STRING,
      anatom_site_1 STRING,
      anatom_site_2 STRING,
      anatom_site_3 STRING,
      anatom_site_4 STRING,
      anatom_site_5 STRING,
      anatom_site_special STRING,
      clin_size_long_diam_mm STRING,
      concomitant_biopsy STRING,
      dermoscopic_type STRING,
      diagnosis_1 STRING,
      diagnosis_2 STRING,
      diagnosis_3 STRING,
      diagnosis_4 STRING,
      diagnosis_5 STRING,
      diagnosis_confirm_type STRING,
      family_hx_mm STRING,
      image_type STRING,
      lesion_id STRING,
      melanocytic STRING,
      patient_id STRING,
      personal_hx_mm STRING,
      sex STRING,
      source_checksum STRING,
      ingestion_run_id STRING,
      ingested_at TIMESTAMP
    )
    USING DELTA
    """
)
dbutils.fs.mkdirs(landing_paths["dataset_archive"])
for path in bronze_paths.values():
    dbutils.fs.mkdirs(path)
for archive in ARCHIVES:
    dbutils.fs.mkdirs(archive["metadata_dir_path"])

display(spark.sql("SHOW TABLES IN bronze"))
display(dbutils.fs.ls(landing_paths["archives"]))
display(dbutils.fs.ls(bronze_paths["root"]))


## Create Silver Objects

Create the shared Silver tables (`silver.image_inventory`, `silver.leakage_groups`, `silver.rejected_records`). These tables are shared across datasets, not ISIC-2019-specific — the `CREATE TABLE IF NOT EXISTS` DDL is safe to run from any dataset's setup notebook.

`silver.image_inventory` is created with only the universal columns first. The label columns (`malignancy`, `specific_diagnosis` — shared by every dataset onboarded so far, `data_platform.labels.SILVER_LABEL_COLUMN_DDL`) are then added via `ALTER TABLE ADD COLUMNS`, guarded in Python rather than SQL `IF NOT EXISTS` (Delta doesn't support that clause on `ADD COLUMNS`), so the table only ever holds the truly shared schema plus whichever datasets' label columns have actually been added — this is the growth mechanism described in `docs/decisions/003-silver-label-columns-not-map.md`, not just a documented intention. A future dataset needing a different label axis (e.g. `severity`) would define its own DDL fragment and run the same kind of guarded `ALTER TABLE` from its own setup notebook.

Silver is table-only per `docs/data_contract.md` (no raw files, only Delta tables), so unlike Bronze this section does not create Volume folders for Silver.


In [ ]:
spark.sql(
    """
    CREATE TABLE IF NOT EXISTS silver.image_inventory (
      dataset_key STRING,
      image_id STRING,
      bronze_uri STRING,
      source_checksum STRING,
      image_width INT,
      image_height INT,
      image_format STRING,
      validation_status STRING,
      validation_reason STRING,
      patient_id STRING,
      lesion_id STRING,
      group_id STRING,
      validated_at TIMESTAMP
    )
    USING DELTA
    """
)

existing_columns = {field.name for field in spark.table("silver.image_inventory").schema.fields}
missing_label_columns = [
    f"{name} {ddl}" for name, ddl in SILVER_LABEL_COLUMN_DDL.items() if name not in existing_columns
]
if missing_label_columns:
    spark.sql(f"ALTER TABLE silver.image_inventory ADD COLUMNS ({', '.join(missing_label_columns)})")

spark.sql(
    """
    CREATE TABLE IF NOT EXISTS silver.leakage_groups (
      dataset_key STRING,
      group_id STRING,
      group_type STRING,
      group_source STRING,
      image_count INT
    )
    USING DELTA
    """
)
spark.sql(
    """
    CREATE TABLE IF NOT EXISTS silver.rejected_records (
      dataset_key STRING,
      image_id STRING,
      bronze_uri STRING,
      rejection_reason STRING,
      rejected_at TIMESTAMP
    )
    USING DELTA
    """
)

display(spark.sql("SHOW TABLES IN silver"))
display(spark.sql("DESCRIBE TABLE silver.image_inventory"))


## Create Gold Objects (placeholder)

Gold is not scoped yet — see `docs/project-checklist.md`. DDL and folder creation for `gold.classification_manifest` and the versioned manifest/preprocessing/dataset-card files will be added here once Gold is designed.


## Upload Checklist

Upload each archive listed in `config/datasets/isic_2019.yaml` to the corresponding path printed above before running `10_bronze_ingest.ipynb`.
